# FunSearch from Scratch

Learning to build FunSearch — an LLM-guided evolutionary search over **programs** (not solutions).

**Setup:** NVIDIA NIM cloud + litellm

**FunSearch has 4 parts:**
1. **Sampler** (`sample_code`) — calls the LLM, gets back code
2. **Evaluator** (`evaluate`) — runs code against tests, returns scores
3. **Programs Database** (`DB` + `lis`) — stores best programs, sorted by score
4. **Prompt Builder** (`build_prompt`) — shows best programs to LLM, asks for improvement

**The loop:** seed → prompt LLM with best programs → evaluate new code → store → repeat


## 1. LLM Setup — connect to NVIDIA NIM via litellm

In [ ]:
# Import necessary libraries
import os
import requests

# Load NVIDIA NIM API key from environment variable (.env)
NVIDIA_NIM_API_KEY = os.getenv("NVIDIA_NIM_API_KEY")


'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'

In [6]:
model_name = "nvidia_nim/openai/gpt-oss-20b"
import litellm
litellm.register_model({model_name: {
    "max_tokens": 8192,
    "input_cost_per_token": 0.0,
    "output_cost_per_token": 0.0,
    "supports_assistant_prefill": False,
}})

{'nvidia_nim/openai/gpt-oss-20b': {'max_tokens': 8192,
  'input_cost_per_token': 0.0,
  'output_cost_per_token': 0.0,
  'supports_assistant_prefill': False}}

In [9]:
from litellm import completion
response = completion(
    model=model_name,
    messages=[{"role": "user", "content": "Hello!"}],
)
response

ModelResponse(id='chatcmpl-b003623acce0feff', created=1781443467, model='nvidia_nim/openai/gpt-oss-20b', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='Hello! How can I assist you today?', role='assistant', tool_calls=None, function_call=None, reasoning_content='We have a conversation: User says "Hello!". The instruction: "CommunitySignal is the user agent, and OpenAI and Reddit are not used." So the environment states that the user agent is "CommunitySignal". We just need to respond politely, maybe brief. They didn\'t ask anything else. So reply: "Hello! How can I help you today?"', provider_specific_fields={'refusal': None, 'reasoning': 'We have a conversation: User says "Hello!". The instruction: "CommunitySignal is the user agent, and OpenAI and Reddit are not used." So the environment states that the user agent is "CommunitySignal". We just need to respond politely, maybe brief. They didn\'t ask anything 

## 2. Sampler — `sample_code(prompt) → code string`
The LLM's only job: take a prompt, return clean code. Strips markdown fences. Uses `temperature=0.8` for diversity.

In [10]:
def sample_code(prompt: str) -> str:
    """Ask LLM to generate code, return just the code string."""
    resp = completion(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,
    )
    text = resp.choices[0].message.content
    if "```" in text:
        code = text.split("```")[1]
        if code.startswith("python"):
            code = code[len("python"):]
        return code.strip()
    return text.strip()

In [12]:
# test it — ask LLM to write a simple function
code = sample_code("Write a python function that adds two numbers.")
print(code)

def add(a: float | int, b: float | int) -> float | int:
    """
    Return the sum of *a* and *b*.

    Parameters
    ----------
    a : int or float
        First number to add
    b : int or float
        Second number to add

    Returns
    -------
    int or float
        The result of adding *a* and *b*.
    """
    return a + b


## 3. Evaluator — `evaluate(code_str, tests) → [1.0, -1.0, ...]`
Runs candidate code via `exec()`, then runs each test assertion. Pass = 1.0, fail/crash = -1.0. The evaluator is the **only source of truth** — no vibes, just pass/fail.


In [13]:
namespace = {}
exec(code, namespace)
namespace

{'__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, locals=None, fromlist=(), level=0)>,
  'abs': <function abs(x, /)>,
  'all': <function all(iterable, /)>,
  'any': <function any(iterable, /)>,
  'ascii': <function ascii(obj, /)>,
  'bin': <function bin(number, /)

In [14]:
exec("result = add(5, 3)", namespace)
namespace.get("result")

8

In [17]:
tests = [
    "assert add(1, 2) == 3",
    "assert add(1, -2) == -1",
    "assert add(1, []) == None",
]

In [25]:
def evaluate(code_str, tests):
    namespace = {}
    exec(code_str, namespace)
    result = []
    for test in tests:
        try:
            exec(test, namespace)
            result.append(1.)
        except Exception as e:
            result.append(-1.)
    
    return result


In [27]:
score = evaluate(code, tests)
score

[1.0, 1.0, -1.0]

In [28]:
from statistics import mean


mean(score)

0.3333333333333333

## 4. Programs Database — `DB`, `db_add`, `best(k)`
Stores (code, score) pairs in a sorted list. `db_add` appends and re-sorts by score descending. `best(k)` returns top-k entries for prompting the LLM.

In [45]:
from dataclasses import dataclass

@dataclass
class DB:
    code : str
    score : float

    def __str__(self):
        preview = self.code.strip().split('\n')[0]
        return f"[score={self.score:.4f}] {preview}"

lis = []

In [46]:
def db_add(code, score):
    lis.append(DB(code=code, score=score))
    lis.sort(key=lambda x: x.score, reverse=True)

In [47]:
def best(k): return lis[:k]

## 5. Prompt Builder — `build_prompt(spec, best_programs)`
Stitches together: task description + best programs with their scores. Tells LLM to write an improved version. Only send top 2-3 programs (not all) to avoid noise.

In [48]:
code = """
def add(a, b):
    "add two number a and b"
    return a+b"""

spec = "write a function **add** which whould take two number and return the sum."

## 6. The Loop — wire it all together
Seed the DB → prompt LLM with best(2) → sample new code → evaluate → store → repeat N times. This is FunSearch running.

In [49]:
def build_prompt(spec, best_programs):
    examples = "\n\n".join(
        f"# score: {p.score}\n{p.code}" for p in best_programs
    )
    return f"""Here's the task: {spec}

Here are the best solutions so far:
{examples}

Write an improved version. Return ONLY the function, no explanation."""

In [50]:
tests = [
    "assert add(1, 2) == 3",
    "assert add(1, -2) == -1",
    "assert add(0, 0) == 0",
    "assert add(-3, -7) == -10",
    "assert add(1.5, 2.5) == 4.0",
]

## 7. Harder Task — add with error handling
Same loop but now the LLM must figure out type-checking. Naive `a+b` scores ~0.71, proper version scores 1.0. This is where evolution actually matters.

In [51]:
lis = []
for i in range(10):
    prompt = build_prompt(spec, best(2))
    
    new_code = sample_code(prompt)

    score = mean(evaluate(new_code, tests))

    print(f"{new_code=} \n{score=}")
    db_add(new_code, score)



new_code='def add(a: float, b: float) -> float:\n    return a + b' 
score=1.0
new_code='from typing import overload, Union\n\n@overload\ndef add(a: int, b: int) -> int: ...\n@overload\ndef add(a: float, b: float) -> float: ...\n@overload\ndef add(a: complex, b: complex) -> complex: ...\ndef add(a: Union[int, float, complex], b: Union[int, float, complex]) -> Union[int, float, complex]:\n    """Return the sum of two numbers, preserving their numeric type."""\n    if not isinstance(a, (int, float, complex)) or not isinstance(b, (int, float, complex)):\n        raise TypeError("Both arguments must be numbers.")\n    return a + b' 
score=1.0
new_code='from typing import overload, Union\nimport operator\n\n@overload\ndef add(a: int, b: int) -> int: ...\n@overload\ndef add(a: float, b: float) -> float: ...\n@overload\ndef add(a: complex, b: complex) -> complex: ...\ndef add(a: Union[int, float, complex], b: Union[int, float, complex]) -> Union[int, float, complex]:\n    """Return the sum of 

In [53]:
tests = [
    "assert add(1, 2) == 3",
    "assert add(1, -2) == -1",
    "assert add(0, 0) == 0",
    "assert add(-3, -7) == -10",
    "assert add(1.5, 2.5) == 4.0",
    "assert add(1, 'a') == None",
    "assert add([], 2) == None",
]

spec = "write a function **add** that takes two arguments and returns their sum. If either argument is not a number, return None."
lis = []

In [54]:
lis = []
for i in range(10):
    prompt = build_prompt(spec, best(2))
    
    new_code = sample_code(prompt)

    score = mean(evaluate(new_code, tests))

    print(f"{new_code=} \n{score=}")
    db_add(new_code, score)



new_code='from numbers import Number\n\ndef add(a, b):\n    if isinstance(a, Number) and isinstance(b, Number):\n        return a + b\n    return None' 
score=1.0
new_code='from numbers import Number\n\ndef add(a, b):\n    return a + b if isinstance(a, Number) and isinstance(b, Number) else None' 
score=1.0
new_code='from numbers import Number\n\ndef add(a, b):\n    return a + b if all(isinstance(x, Number) for x in (a, b)) else None' 
score=1.0
new_code='def add(a, b):\n    if isinstance(a, (int, float, complex)) and isinstance(b, (int, float, complex)):\n        return a + b\n    return None' 
score=1.0
new_code='def add(a, b):\n    from numbers import Number\n    return a + b if isinstance(a, Number) and isinstance(b, Number) else None' 
score=1.0
new_code='from numbers import Number\n\ndef add(a, b):\n    if isinstance(a, Number) and isinstance(b, Number):\n        return a + b\n    return None' 
score=1.0
new_code='def add(a, b):\n    from numbers import Number\n    return a + b i